In [6]:
from pathlib import Path
import pandas as pd

from trino_stack.lakehouse import Lakehouse
from analysis import analyse_workload_set


# ---------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------

WORKLOAD_ROOT = Path("/mnt/primary/Main/Workloads")
RESULTS_DIR = Path("results")

# All TPC-DS baseline workloads (every "*tpcds*" dir under WORKLOAD_ROOT),
# except gpt_generated_tpcds -- that one has 1000 queries vs ~100 for the
# rest, so it isn't a fair comparison here.
WORKLOAD_NAMES = [
    "gpt_generated_1000_tpcds_v7",
]

CATALOG = "iceberg"
SCHEMA = "tpcds"


# ---------------------------------------------------------------------
# Run standardised analysis
# ---------------------------------------------------------------------
#
# For every workload in WORKLOAD_NAMES this produces one row combining:
#   - live Trino plan-diversity metrics (table/column/join coverage,
#     entropy, plan uniqueness, Vendi score, ...)
#   - "meta_"-prefixed static SQL metaheuristics (table/join/aggregation
#     counts, complexity mix, table-usage entropy, schema coverage, ...),
#     taken from generation_report.json when the workload has one, and
#     computed on the fly from the .sql files otherwise (e.g. "tpcds").
#
# Any workload directory (name under WORKLOAD_ROOT, or an absolute path)
# works here, generated or hand-written alike.

lh = Lakehouse.from_release(
    instance_name="lakehouse-g",
    namespace="pgr24james",
    sync_schemas=True,
    verbose=False,
)

overview_df, failures_df = analyse_workload_set(
    WORKLOAD_NAMES,
    workload_root=WORKLOAD_ROOT,
    lh=lh,
    catalog=CATALOG,
    schema=SCHEMA,
)

overview_df

overview_df.to_csv(
    RESULTS_DIR / "gpt_generated_1000_tpcds_v7.csv",
    index=False,
)

if not failures_df.empty:
    print("\n=== Failures ===")
    display(failures_df)

Checking Trino health ...

=== Analysing workload: gpt_generated_1000_tpcds_v7 ===
                   workload  sql_queries  planned_queries  table_coverage_pct  column_coverage_pct  join_edge_coverage_pct  table_usage_entropy  column_usage_entropy  join_edge_usage_entropy  unique_table_set_ratio  unique_plan_graphs  plan_uniqueness_ratio  operator_types_observed  operator_type_entropy  unique_operator_instances  operator_ngram_entropy  mean_nn_plan_distance  min_nn_plan_distance  plan_graph_vendi_score
gpt_generated_1000_tpcds_v7         1000             1000               100.0                99.06                   100.0               0.9909                0.8885                   0.9367                   0.918                1000                    1.0                       30                 0.6767                      17654                  0.5771                 0.2225                0.0172                  54.142
